In [ ]:
print('hi')

Okay so the final project ! 
and man this is complex or in the starting feels very so
dont know how oauth will work, how to connect to gmail/calendar -
how to do integrations 3rd party
how the state will flow and all
hopefully these gets answered as the project goes.
I will do it !

not clear or sure about the architecture as well
how many agents do we take?
one to clasify the intent.
one to actually schedule the time slot when we get info from calendar?
one to draft a reply?
or we shall not create many as it is unnessacary inc in no of agents?
what do we take in state?

In [ ]:
#structured output for extraction
from typing import Optional
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv

class meetingDetails(TypedDict):
    sender_name : Optional[str] = 'No Name Provided'
    sender_email_address : str
    agenda : str
    requested_day : list[str]
    time_preference : str
    time_duration : int

In [ ]:
#Structured output for general agent
from typing import Literal
class Intent(TypedDict) : 
    intent : Literal['Meeting_Email', 'Not_Meeting_Email']

In [ ]:
#Structured Output for draft email
class draft_email_SO(TypedDict) : 
    subject : str 
    body : str

In [ ]:
#defining the state

class Fresh_Email_State(TypedDict):
    messages : Annotated[list, add_messages]
    intent : str 
    extracted_email_info : meetingDetails 
    needs_clarification : bool 
    available_slots : dict 
    draft_response : draft_email_SO 
    recommended_slots : list
    draft_email_approved : bool
    thread_id : str
    

In [ ]:
#classifier LLM
load_dotenv(override=True)
classify_llm = ChatOpenAI(model='gpt-5')
classify_llm_with_so = classify_llm.with_structured_output(Intent)

In [ ]:
def classify_email_agent(state : Fresh_Email_State) :
    print('inside classify email agent ...')
    SYSTEM_PROMPT = f"""You are an expert triage agent classifying email intent.

    [CRITICAL INSTRUCTIONS]
    Categorize the incoming email into exactly one of these two intents:
    - 'Meeting_Email': Select this if the sender is asking, proposing, or suggesting a meeting, interview, sync, phone call, calendar slot, or scheduling an explicit block of time (even if phrased casually like "available for a meeting on...", "free tomorrow?", or "let's connect").
    - 'Not_Meeting_Email': Select this if the email is about anything else (e.g., support tickets, updates, general info, spam, newsletters).

    Analyze carefully: A casual invitation like "available on 26th June around 11 PM?" IS A MEETING EMAIL.
    Here's the email - \n {state['messages']}
    """
    print(state['messages'])
    print()
    response = classify_llm_with_so.invoke(
        input=[SystemMessage(content=SYSTEM_PROMPT)] 
    )
    print(response)
    return {
        'intent' : response['intent']
    }

In [ ]:
def post_classication_routing(state : Fresh_Email_State) :
    print('inside classification routing...')
    print(state['intent']) 
    return 'handle_meeting_requests' if state['intent']=='Meeting_Email' else 'Not_Meeting_Email'

In [ ]:
def no_meeting_node(state : Fresh_Email_State) :
    print('inside no_meeting_node') 
    return { 
        'messages' : 'This is not a meeting request E-mail. So no action required from me.' 
    }

In [ ]:
extract_llm = ChatOpenAI(model='gpt-5-nano')
extract_llm_with_SO = extract_llm.with_structured_output(meetingDetails)

In [ ]:
def handle_meeting_requests(state : Fresh_Email_State) :
    PROMPT = f''' 
    You are an agent whose work is to extract important details from e-mails. 
    === DETERMINISTIC EMAIL METADATA ===
    Sender Info: {state["extracted_email_info"].get("sender_email_address", "Not Provided")}
    threadId : {state["thread_id"]}
     === EMAIL CONTENT BODY ===
    Here's the email - {state["messages"][-1].content}
    When you are provided with an e-mail,
    you have to extract things like :-
    - sender's name(if provided in email)
    - sender's email address
    - agenda - the agenda/motive for the meeting
    - requested day - the day they're proposing the meeting for
    - time-preference - their preferred/proposed time slot - The normalized target time extracted from the email in strict '%I:%M %p' format (e.g., '04:00 PM', '11:30 AM').
                        If vague like 'somewhere near to 11 AM', normalize it to the exact top of the hour like '11:00 AM'."
    - time_duration - the duration of the meeting
    If the email contains non-specific information such as  :-
    "Any time next week"
    "Sometime after the holiday" 
    "Whenever you're free"
    "June 25-27" 
    then make it as requires more clarification.
    '''

    response = extract_llm_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(response)
    return {
        'extracted_email_info' : response
    }

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta
from calendar_service import list_upcoming_events

WORK_START_HOUR = 10
WORK_END_HOUR = 19
MEETING_DURATION = 30


def get_available_slots(state:Fresh_Email_State):
    print('inside get_available_slots ...')
    events = list_upcoming_events()
    events_by_day = defaultdict(list)
    for event in events:
        start = datetime.fromisoformat(event["start"]["dateTime"])
        end = datetime.fromisoformat(event["end"]["dateTime"])
        events_by_day[start.date()].append((start, end))
    
    print(f'refactored events_by_day - {events_by_day}') 
    available_slots = {}
    for day, busy_slots in events_by_day.items():
        busy_slots.sort(key=lambda x: x[0])
        tz = busy_slots[0][0].tzinfo
        
        work_start = datetime.combine(
            day,
            datetime.min.time()
        ).replace(
            hour=WORK_START_HOUR,
            tzinfo=tz
        )

        work_end = datetime.combine(
            day,
            datetime.min.time()
        ).replace(
            hour=WORK_END_HOUR,
            tzinfo=tz
        )

        cursor = work_start
        day_slots = []
        for event_start, event_end in busy_slots:
            while cursor + timedelta(minutes=MEETING_DURATION) <= event_start:
                day_slots.append(
                    (
                        cursor.strftime("%Y-%m-%d %H:%M"),
                        (cursor + timedelta(minutes=MEETING_DURATION)).strftime("%Y-%m-%d %H:%M")
                    )
                )
                cursor += timedelta(minutes=MEETING_DURATION)
            cursor = max(cursor, event_end)

        while cursor + timedelta(minutes=MEETING_DURATION) <= work_end:
            day_slots.append(
                (
                    cursor.strftime("%Y-%m-%d %H:%M"),
                    (cursor + timedelta(minutes=MEETING_DURATION)).strftime("%Y-%m-%d %H:%M")
                )
            )
            cursor += timedelta(minutes=MEETING_DURATION)
        available_slots[str(day)] = day_slots
    print(f'returning available slots - {available_slots}')
    print(state["extracted_email_info"]["time_preference"] ) 
    return {
        'available_slots' : available_slots 
    }  

In [ ]:
from langchain_openai import ChatOpenAI 
draft_email_llm = ChatOpenAI(model='gpt-5-nano') 
draft_email_llm_with_SO = draft_email_llm.with_structured_output(draft_email_SO)

In [ ]:
def select_best_slots(available_slots, date, preferred_time):
    print('inside select_best_slots ...')
    slots = available_slots.get(date, [])
    target_time = datetime.strptime(
        preferred_time,
        "%I:%M %p"
    )
    ranked = []
    for start, end in slots:
        start_dt = datetime.strptime(
            start,
            "%Y-%m-%d %H:%M"
        )
        diff = abs(
            (start_dt.hour * 60 + start_dt.minute)
            - (target_time.hour * 60 + target_time.minute)
        )
        ranked.append((diff, start, end))
    ranked.sort(key=lambda x: x[0])
    return ranked[:3]

In [ ]:
def select_best_slots_node(state: Fresh_Email_State):
       
    recommended_slots = select_best_slots(
        available_slots=state["available_slots"],
        date="2026-06-21",      # temporarily hardcoded 
        preferred_time=state["extracted_email_info"]["time_preference"] 
    )
    print(f'recommended slots - {recommended_slots}')
    return {
        "recommended_slots": recommended_slots
    }

In [ ]:
def draft_reply(state : Fresh_Email_State) :
    print('inside draft_reply ...')
    PROMPT = f''' 
    You are working as a a personal e-mail drafting assistant for me. You are provided with a proposed meeting details
    and also, my available time slots. You have to draft a reply to the proposer, informing about the time slots which
    I will be available on, respond with the available time slots which are near the proposed meeting time. Respond with a maximum of 
    3-5 slots. 
    here arw the meetiong details : \n
    {state['extracted_email_info']} \n
    and here are my available slots \n
    {state['recommended_slots']}
    '''
    response = draft_email_llm_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(response)
    return {
        'draft_response' : response
    }
    

In [ ]:
from langgraph.types import interrupt, Command

def hITL_node(state: Fresh_Email_State) :
    '''implements HITL'''
    print('inside HITL Node')
    approval = interrupt(f"Here\'s the draft mail - \n {state["draft_response"]} \n Approve? Yes/No...")
    print(approval) 
    if approval.lower()=='yes':
        return {
            'draft_email_approved' : True
        }
    else:
        return {'draft_email_approved': False}

In [ ]:
from gmail_service import send_message

def send_email_node(state : Fresh_Email_State) :
    recipient = state['extracted_email_info']['sender_email_address']
    subject = state['draft_response']['subject']
    body = state['draft_response']['body'] 
    print('Sending email now ...') 
    result = send_message(recipient, subject, body)
    print(result)
    return  {
        'thread_id' : result['threadId']
    }
    
    

In [ ]:
def email_not_approved_node(state : Fresh_Email_State) : 
    return {
        'messages' : ''' Draft email has not been approved ...... We need to work on this a bit more !'''
        }

In [ ]:
def approval_node_routing(state: Fresh_Email_State) :
    return 'send_email_action' if state['draft_email_approved'] else 'email_not_approved_action'

In [ ]:
import json
from pathlib import Path
import sqlite3, os 
from datetime import datetime

folder_path = Path(r'C:\AppyProjects\CustomGenAIProjects\ai_meeting_coordination_agent\email_data')

def store_email_data_db_node(state : Fresh_Email_State) :
    proposed_slots = json.dumps(state['recommended_slots'])
    conn = sqlite3.connect(folder_path/'email_data.db')
    cursor = conn.cursor()
    cursor.execute(
        (''' 
        INSERT INTO scheduling_conversations (
            threadId,    
            recipient_address, 
            proposed_slots,       
            status,         
            created_at,
            agenda              
        ) VALUES (?,?,?,?,?,?)
        '''
        ),
        (
            state['thread_id'], state['extracted_email_info']['sender_email_address'], proposed_slots,
            'Waiting_For_Reply', datetime.now().isoformat(), state['extracted_email_info']['agenda']
        )
    )
    conn.commit()
    #conn.close()
    print('Record saved successfully !')
    

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

graph_builder = StateGraph(Fresh_Email_State)

graph_builder.add_node('classify_email_agent', classify_email_agent)
graph_builder.add_node('no_meeting_node', no_meeting_node)
graph_builder.add_node('handle_meeting_requests', handle_meeting_requests)
graph_builder.add_node('get_available_slot',get_available_slots)
graph_builder.add_node('select_best_slots_node',select_best_slots_node)
graph_builder.add_node('draft_reply',draft_reply) 
graph_builder.add_node('approval_node',hITL_node)
graph_builder.add_node('send_email_node',send_email_node)
graph_builder.add_node('email_not_approved_node',email_not_approved_node)
graph_builder.add_node('store_email_data_db_node',store_email_data_db_node)

graph_builder.add_conditional_edges('classify_email_agent', post_classication_routing, {'Not_Meeting_Email':'no_meeting_node', 'handle_meeting_requests':'handle_meeting_requests'})
graph_builder.add_edge(START, 'classify_email_agent')
graph_builder.add_edge('handle_meeting_requests', 'get_available_slot')
graph_builder.add_edge('get_available_slot', 'select_best_slots_node')
graph_builder.add_edge('select_best_slots_node', 'draft_reply')
graph_builder.add_edge('draft_reply', 'approval_node')
graph_builder.add_conditional_edges('approval_node', approval_node_routing, {'send_email_action' : 'send_email_node', 'email_not_approved_action' : 'email_not_approved_node'})
graph_builder.add_edge('send_email_node', 'store_email_data_db_node')

memory = MemorySaver()
fresh_email_graph = graph_builder.compile(checkpointer=memory)


In [ ]:
from IPython.display import display, Image
#display(Image(fresh_email_graph.get_graph().draw_mermaid_png()))

WORKFLOW PART 2

WHEN REPLY COMES THEN ....a new graph execution starts so...

In [ ]:
#slot SO
class SelectedSlot(TypedDict):
    slot_id: int
    start_time: str
    end_time: str

In [ ]:
#structured output for reply email data extration
class ReplyExtraction(TypedDict) : 
    selected_slot : Optional[SelectedSlot] 
    explanation : Optional[str] 
    intent : Literal['Accepted_slot', 'Rejected_slot', 'Request Reschedule','Needs Clarification']

In [ ]:
class ConfirmationState(TypedDict) :
    reply_email : str 
    replyEmailExtractedDetails : ReplyExtraction 
    conversationData:str 
    schedule_meeting : bool

In [ ]:
reply_extraction_llm = ChatOpenAI(model='gpt-5-nano') 
reply_extraction_llm_with_so = reply_extraction_llm.with_structured_output(ReplyExtraction)

In [ ]:
def dont_schedule_node(replyState : ConfirmationState) :
    return{
        'CHILLLLLLL' 
    }

In [ ]:
def extractReplyEmailData (replyState : ConfirmationState) : 
    print('inside extractReplyEmailData...')
    SYSTEM= f''' You are provided an email and you have to extract important information from it so that the ceo can know 
    what is the other party's response to the original email was. 
    This is the received email - {replyState['reply_email']} 
    These were the proposed slots:
    {replyState['conversationData']['proposed_slots']}

    If the user accepts one of the proposed slots,
    return the EXACT slot from the list.
    ''' 
    response = reply_extraction_llm_with_so.invoke( 
        input=[SystemMessage(content=SYSTEM)] 
    ) 
    print(response) 
    return {
        'replyEmailExtractedDetails' : response 
    }

In [ ]:
def approve_meeting_node(replyState:ConfirmationState) :   
    print('inside approve_meeting_node') 
    approve_meeting = f''' 
        {replyState['conversationData']['recipient_address']} has proposed a meeting for {replyState['conversationData']['agenda']} 
        We had given them these available slots - \n {replyState['conversationData']["proposed_slots"]} and he/she has chosen \n
        {replyState['replyEmailExtractedDetails']['selected_slot']} 
        Are you okay with me to schedule a meeting at the confirmed time? Pausing state for input !
        '''
    print(approve_meeting) 

    create_event = interrupt('Schedule Meeting ?  Yes/No...') 
    if create_event.lower() == 'yes': 
        return { 'schedule_meeting' : "Go_ahead" } 
    else:
        return { 'schedule_meeting' : "dont_schedule" }

In [ ]:
def schedule_routing(replyState : ConfirmationState) : 
    return "create_event" if replyState["schedule_meeting"]=='Go_ahead' else 'dont_schedule' 

In [ ]:
#we need conditional routing here for the non-happy flow. maybe in V2 

In [ ]:
from calendar_service import create_event 

def create_event_node(replyState : ConfirmationState) : 
    print('inside create_event_node...')
    summary = replyState['conversationData']['agenda'] 
    location = 'Google Meet' 
    attendees = replyState['conversationData']['recipient_address'] 
    print('HELOOOO') 
    #selected_slot = "2026-06-20 17:00-17:30"
    start_time = replyState['replyEmailExtractedDetails']['selected_slot']['start_time'] 
    end_time = replyState['replyEmailExtractedDetails']['selected_slot']['end_time'] 
    print(start_time) 
    print(f'end time - {end_time}')
    create_event(summary, location,start_time, end_time,attendees)
    
    

In [ ]:
import sqlite3
from pathlib import Path
folder_path = Path(r'C:\AppyProjects\CustomGenAIProjects\ai_meeting_coordination_agent\email_data')

conn = sqlite3.connect(folder_path/'email_data.db') 
cursor = conn.cursor() 
cursor.execute(
    ''' 
    select * from scheduling_conversations where threadId = ? 
    ''',('19ef973ebf77851a',) 
)
res = cursor.fetchone()
print(res)

In [ ]:
def update_email_data_in_db(replyState : ConfirmationState) :
    print('inside update_email_data_in_db.... ')
    conn = sqlite3.connect(folder_path/'email_data.db') 
    cursor = conn.cursor() 
    print(replyState)
    print(f'test - {replyState['conversationData']['threadId']}')
    cursor.execute(
        ''' 
        UPDATE scheduling_conversations
        SET status = ?
        WHERE threadId = ?
        ''' , ('Event Created', replyState['conversationData']['threadId']) 
    )
    conn.commit()
    print('\n Status has been updated !')
    

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(ConfirmationState)

graph_builder.add_node('extractReplyEmailData', extractReplyEmailData)
#graph_builder.add_node('fetch_conversation_from_db', fetch_conversation_from_db)
graph_builder.add_node('approve_meeting', approve_meeting_node)
graph_builder.add_node('create_event_node', create_event_node) 
graph_builder.add_node('dont_schedule_node', dont_schedule_node)
graph_builder.add_node('update_email_data_in_db', update_email_data_in_db)

# graph_builder.add_edge(START, 'fetch_conversation_from_db')
# graph_builder.add_edge('fetch_conversation_from_db', 'extractReplyEmailData')
graph_builder.add_edge(START, 'extractReplyEmailData')
graph_builder.add_edge('extractReplyEmailData', 'approve_meeting') 
graph_builder.add_conditional_edges('approve_meeting', schedule_routing, {'create_event' : 'create_event_node', 'dont_schedule':'dont_schedule_node'})
graph_builder.add_edge('create_event_node', 'update_email_data_in_db')

memory = MemorySaver()
confirmation_Graph = graph_builder.compile(memory)


In [ ]:
from IPython.display import Image, display 
#display(Image(confirmation_Graph.get_graph().draw_mermaid_png())) 


In [ ]:

import time
import uuid
from langchain_core.messages import HumanMessage
from gmail_service import fetch_new_mails, mark_email_as_read

while True:
    fetched_emails = fetch_new_mails()
    
    print()
    for mail in fetched_emails: 
        
        mail_payload = mail['email_data']
        messageId = mail_payload['id'] 
        mail_threadId = mail_payload['threadId'] 
        mail_content = mail_payload.get('snippet', '') 
        
        headers = mail_payload['payload']['headers']
        sender_email = next((h['value'] for h in headers if h['name'].lower() == 'from'), "Unknown Sender")
        
        config = {'configurable' : {'thread_id' : f'mail_session{mail_threadId}'}}
        
        if mail['graph_type'] == 'confirmation': 
            
            print('invoking confirmation graph...') 
            try : 
                confirmation_Graph.invoke({ 
                    'reply_email' : mail_content, 
                    'conversationData' : dict(mail['db_record'])
                },config)
                
                state = confirmation_Graph.get_state(config)
                if state.next:
                    approval = input('HELOOOOO !Shall I create an event at the confirmed time slot?') 
                    confirmation_Graph.invoke(Command(resume=approval), config) 
                    mark_email_as_read(messageId)
                    print('Marked mail as read')
            except Exception as e :
                print(f'Exception occured ! - {e}')
        else: 
            print('invoking fresh email graph...') 
            try:
                fresh_email_graph.invoke({ 
                    'messages' : [HumanMessage(content=mail_content)],
                    'thread_id' : mail_threadId,
                    'extracted_email_info' : {
                        'sender_email_address' : sender_email
                    }
                    }, config)  
                state = fresh_email_graph.get_state()
                if state.next:
                    approval = input('Approve Draft? Yes/no ? ......')
                    fresh_email_graph.invoke(Command(resume=approval), config)
                    mark_email_as_read(messageId)
                    print('Marked mail as read')
            except Exception :
                print('Exception occured !')
    print() 
    time.sleep(20)
